In [2]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.express as px

In [3]:
# To make plotly fig show in notebook
import plotly.io as pio
pio.renderers.default = "notebook"

# MAIN

In [98]:
# Read filtered TSV (only non fully '0' rows)
X = pd.read_csv("CUSTOM_hg38_episign/meth_matrix.tsv", sep="\t", index_col=0)

# Compute epiSize (= size of epiSign):
epiSize = {}
for epiSign in [x for x in X.columns if x not in ('coord')]:
    epiSize[epiSign] = sum(X[epiSign] > 99)  # Catch 100% methyl

In [99]:
# Transpose
X_t = X.T  # Required
print(X_t.index)

to_PCA = X_t
if 'epiSize' in X_t.columns:
    to_PCA = X_t.drop('epiSize', axis=1)

# Remove 2nd row = size of epiSignormalize) then normalize
X_scaled = StandardScaler().fit_transform(to_PCA)

Index(['ADCADN.bed', 'ATRX.bed', 'AUTS18.bed', 'BAFopathy.bed', 'BFLS.bed',
       'CHARGE.bed', 'CdLS.bed', 'Down.bed', 'Dup7.bed', 'EEOC.bed',
       'FLHS.bed', 'GTPTS.bed', 'HMA.bed', 'HVDAS_C.bed', 'HVDAS_T.bed',
       'ICF1.bed', 'ICF2_3_4.bed', 'KDVS.bed', 'Kabuki.bed', 'Kleefstra.bed',
       'MRD51.bed', 'MRX93.bed', 'MRX97.bed', 'MRXCJS.bed', 'MRXSN.bed',
       'MRXSSR.bed', 'RMNS.bed', 'RSTS.bed', 'SBBYSS.bed', 'SETD1B.bed',
       'Sotos.bed', 'TBRS.bed', 'WDSTS.bed', 'Williams.bed', 'HG002_combined',
       'barcode04_combined'],
      dtype='object')


In [100]:
# Run PCA:
NB_COMPON = 3
pca = PCA(n_components=NB_COMPON)
pcs = pca.fit_transform(X_scaled)

# Make a dict with '% variance explained' for each component:
dict_compon = {'compon'+str(i) : str(round(pca.explained_variance_ratio_[i]*100,4)) for i in range(NB_COMPON)}

In [101]:
# Top N features of each componennt
compon_0_top = np.abs(pca.components_[0]).argsort()[::-1][:5]
print("Component 0:", list(X.index[compon_0_top]))

compon_1_top = np.abs(pca.components_[1]).argsort()[::-1][:5]
print("Component 1:", list(X.index[compon_1_top]))

Component 0: ['10:130045533-130045534', '7:4308119-4308120', '3:11705290-11705291', '19:7615166-7615167', '19:15021371-15021372']
Component 1: ['9:19379119-19379120', '1:2577832-2577833', '6:128492173-128492174', '2:233355516-233355517', '16:85812204-85812205']


In [102]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
pcs_df = pd.DataFrame(
    pcs,
    index=X.columns,
    columns=dict_compon.keys()
)
print(pcs_df.loc[['Kabuki.bed', 'barcode04_combined', 'HG002_combined']])

                      compon0    compon1    compon2
Kabuki.bed          -4.009820  59.879305  24.078432
barcode04_combined   6.920526  -3.100686  -0.981972
HG002_combined      87.782472   2.075491  -0.560141


In [103]:
# Plot PCA
x_compon = 'compon0'
y_compon = 'compon1'

fig = px.scatter(
        pcs_df,
        x=x_compon,
        y=y_compon,
        hover_data=[pcs_df.index],
        color=epiSize,
        labels={x_compon:':'.join([x_compon,dict_compon[x_compon]]), y_compon:':'.join([y_compon,dict_compon[y_compon]])}
)
fig.show()